In [1]:
import pandas as pd
import numpy as np
import re
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

In [2]:
INPUT_XLSX = r"Vulnerabilities In Scope 05-Jan-26 (1).xlsx"
OUTPUT_XLSX = r"Vulnerabilities In Scope 05-Jan-26 (1).xlsx"
NEW_SHEET_NAME = "AssetGroups_ByExactCVEset"

In [3]:
# --- Columns expected from Wiz export (based on file preview) ---
USECOLS = [
    "Name",                       # CVE / vuln name
    "AssetName",                  # system / app identifier
    "Category",                   # Wiz category (often tech grouping)
    "AssetType",                  # e.g., VIRTUAL_MACHINE, SERVERLESS, CONTAINER_IMAGE
    "CVE Severity",               # Critical/High/Medium/Low
    "HasCisaKevExploit",          # KEV indicator (boolean)
    "Asset has wide internet exposure",   # direct exposure
    "Asset has limited internet exposure" # optional, if you want it
]

In [4]:
# --- Target type split columns (the category buckets) ---
TYPE_COLUMNS = [
    "AKS_Cluster",
    "Application",
    "Container Image",
    "Container",
    "Function App",
    "Infra asset",
    "Matillion",
    "MS_Managed_Lance Asset",
    "Security_Asset",
    "VM",
    "VMSS",
    "ZScaler"
]

In [5]:
def norm_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def norm_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    return s in ("true", "1", "yes", "y")

def classify_asset(row) -> str:
    """
    Best-effort classification into the 12 requested buckets.
    Uses Category first (more specific), then AssetType.
    """
    cat = norm_str(row.get("Category", "")).lower()
    at  = norm_str(row.get("AssetType", "")).upper()

    # AKS / Kubernetes
    if "aks" in cat or "kubernetes" in cat:
        return "AKS_Cluster"

    # ZScaler
    if "zscaler" in cat:
        return "ZScaler"

    # Matillion
    if "matillion" in cat:
        return "Matillion"

    # Security assets (best-effort keywording)
    if "security" in cat or "sentinel" in cat or "defender" in cat or "vault" in cat:
        return "Security_Asset"

    # Microsoft-managed assets/images (best-effort keywording)
    if "ms managed" in cat or "microsoft-managed" in cat or "microsoft managed" in cat:
        return "MS_Managed_Lance Asset"

    # Container Image
    if "image" in cat or at == "CONTAINER_IMAGE":
        return "Container Image"

    # Container runtime / workload object (if present in Category)
    if "container" in cat and "image" not in cat:
        return "Container"

    # VMSS
    if "vmss" in cat or "scale set" in cat:
        return "VMSS"

    # VM
    if "vm" in cat or at == "VIRTUAL_MACHINE":
        return "VM"

    # Function Apps (serverless)
    if "function" in cat or at == "SERVERLESS":
        return "Function App"

    # Applications (web apps, app services, etc.)
    if "app" in cat or "application" in cat or "web" in cat:
        return "Application"

    # Default fallback
    return "Infra asset"

def severity_bucket(sev: str) -> str:
    """
    You requested Critical / Medium / Low buckets.
    Wiz often has High as well; we place High into 'Critical' bucket by default.
    Adjust if you want High treated differently.
    """
    s = norm_str(sev).lower()
    if s in ("critical", "high"):
        return "Critical"
    if s == "medium":
        return "Medium"
    if s == "low":
        return "Low"
    # Unknown / blank -> Low (conservative for grouping display)
    return "Low"

In [6]:
# --- Load data (only required columns) ---
df = pd.read_excel(INPUT_XLSX, engine="openpyxl", usecols=USECOLS)

# Normalize
df["AssetName"] = df["AssetName"].astype("string").fillna("").map(norm_str)
df["Name"] = df["Name"].astype("string").fillna("").map(norm_str)
df["CVE Severity"] = df["CVE Severity"].astype("string").fillna("").map(norm_str)

df["HasCisaKevExploit"] = df["HasCisaKevExploit"].map(norm_bool)
df["Asset has wide internet exposure"] = df["Asset has wide internet exposure"].map(norm_bool)
df["Asset has limited internet exposure"] = df["Asset has limited internet exposure"].map(norm_bool)

# Drop unusable rows
df = df[(df["AssetName"] != "") & (df["Name"] != "")].copy()

# Add derived fields
df["AssetTypeBucket"] = df.apply(classify_asset, axis=1)
df["SevBucket"] = df["CVE Severity"].map(severity_bucket)

# De-duplicate rows at (Asset, CVE) level to avoid inflated lists/counts
df_pairs = df.drop_duplicates(subset=["AssetName", "Name"]).copy()

# --- Build per-asset CVE set (canonical tuple) ---
asset_to_cves = (
    df_pairs.groupby("AssetName")["Name"]
    .apply(lambda s: tuple(sorted(set(s))))
    .to_dict()
)

# Map each asset to its canonical CVE-set key
assets = pd.DataFrame({"AssetName": list(asset_to_cves.keys())})
assets["CVE_Set_Key"] = assets["AssetName"].map(asset_to_cves)

# Group assets by identical CVE set
grouped_assets = assets.groupby("CVE_Set_Key")["AssetName"].apply(lambda s: sorted(s.tolist())).reset_index()
grouped_assets.rename(columns={"AssetName": "Systems"}, inplace=True)

# Helper: quick lookups
# For any CVE, determine its severity bucket (if multiple severities exist, take the "worst")
sev_rank = {"Critical": 3, "Medium": 2, "Low": 1}
cve_to_sev = (
    df_pairs.groupby("Name")["SevBucket"]
    .apply(lambda s: max(set(s), key=lambda x: sev_rank.get(x, 1)))
    .to_dict()
)

cve_is_kev = df_pairs.groupby("Name")["HasCisaKevExploit"].max().to_dict()

# For any (asset), is directly internet exposed?
asset_direct_exposed = (
    df_pairs.groupby("AssetName")["Asset has wide internet exposure"].max().to_dict()
)

# If you want "direct" to include limited exposure too, replace the above with:
# asset_direct_exposed = df_pairs.groupby("AssetName")[["Asset has wide internet exposure", "Asset has limited internet exposure"]].max().max(axis=1).to_dict()

# Asset -> type bucket
asset_type_bucket = (
    df_pairs.groupby("AssetName")["AssetTypeBucket"]
    .agg(lambda s: sorted(set(s))[0] if len(set(s)) else "Infra asset")
    .to_dict()
)

In [7]:
# --- Build output rows ---
out_rows = []
for idx, row in grouped_assets.iterrows():
    systems = row["Systems"]
    sys_count = len(systems)

    cves = list(row["CVE_Set_Key"])
    cve_count = len(cves)

    # Severity lists
    critical_cves = sorted([c for c in cves if cve_to_sev.get(c, "Low") == "Critical"])
    medium_cves   = sorted([c for c in cves if cve_to_sev.get(c, "Low") == "Medium"])
    low_cves      = sorted([c for c in cves if cve_to_sev.get(c, "Low") == "Low"])

    # KEVs
    kev_cves = sorted([c for c in cves if cve_is_kev.get(c, False)])

    # Internet-exposed systems list
    exposed_systems = sorted([a for a in systems if asset_direct_exposed.get(a, False)])

    # Type splits: list assets in each requested bucket
    type_lists = {t: [] for t in TYPE_COLUMNS}
    for a in systems:
        t = asset_type_bucket.get(a, "Infra asset")
        if t not in type_lists:
            t = "Infra asset"
        type_lists[t].append(a)
    for t in TYPE_COLUMNS:
        type_lists[t] = ", ".join(sorted(type_lists[t]))

    out_rows.append({
        "System Group (same CVE set)": ", ".join(systems),
        "System Count": sys_count,
        "Common CVEs": ", ".join(cves),
        "CVE Count": cve_count,
        "Critical CVEs": ", ".join(critical_cves),
        "Medium CVEs": ", ".join(medium_cves),
        "Low CVEs": ", ".join(low_cves),
        "Critical Count": len(critical_cves),
        "Medium Count": len(medium_cves),
        "Low Count": len(low_cves),
        "KEV CVEs": ", ".join(kev_cves),
        "Direct Internet Exposed Systems": ", ".join(exposed_systems),
        **type_lists
    })

out_df = pd.DataFrame(out_rows)

# Optional: sort largest groups first
out_df = out_df.sort_values(by=["System Count", "CVE Count"], ascending=[False, False]).reset_index(drop=True)

In [8]:
# --- Write to a NEW worksheet in the same workbook ---
wb = load_workbook(OUTPUT_XLSX)

# Remove existing sheet if present (so reruns are clean)
if NEW_SHEET_NAME in wb.sheetnames:
    ws_old = wb[NEW_SHEET_NAME]
    wb.remove(ws_old)

ws = wb.create_sheet(NEW_SHEET_NAME)

for r in dataframe_to_rows(out_df, index=False, header=True):
    ws.append(r)

In [9]:
# Basic formatting: wrap long columns
from openpyxl.styles import Alignment, Font
wrap_cols = ["A", "C", "E", "F", "G", "K", "L"]  # key list fields
for col in wrap_cols:
    for cell in ws[col]:
        cell.alignment = Alignment(wrap_text=True, vertical="top")

# Bold header
for cell in ws[1]:
    cell.font = Font(bold=True)

wb.save(OUTPUT_XLSX)

print(f"Created worksheet '{NEW_SHEET_NAME}' in {OUTPUT_XLSX}")

Created worksheet 'AssetGroups_ByExactCVEset' in Vulnerabilities In Scope 05-Jan-26 (1).xlsx
